In [ ]:
# Import necessary libraries
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

train = pd.read_csv("./application_train_random_combination.csv")
X_train = train.drop(columns=['TARGET'])
y_train = train['TARGET']

test = pd.read_csv("./application_train_test.csv")
X_test = test.drop(columns=['TARGET'])
y_test = test['TARGET']

valid = pd.read_csv("./application_train_valid.csv")
X_valid = test.drop(columns=['TARGET'])
y_valid = test['TARGET']

In [ ]:
# Step 1: Standardize the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

# Step 2: Perform PCA
pca = PCA()
X_pca = pca.fit_transform(X_train)

# Step 3: Analyze Explained Variance
explained_variance = pca.explained_variance_ratio_
cumulative_variance = explained_variance.cumsum()

# Plot cumulative explained variance
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='--')
plt.title('Cumulative Variance Explained by Principal Components')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.grid()
plt.show()

loss_data = {}

# Display the explained variance and dimensionality loss
for i, cumulative in enumerate(cumulative_variance):
    if cumulative not in loss_data:
        loss_data[cumulative] = i+1

In [ ]:
from collections import Counter

# Assuming y_test is a list or numpy array
counter = Counter(y_train)
print(counter)

In [ ]:
import lightgbm as lgb
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, roc_curve
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Define the objective function for Optuna
def objective(trial):
    # Define the hyperparameter search space
    param = {
        'objective': 'binary',
        'metric': 'auc',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'max_depth': trial.suggest_int('max_depth', 5, 50),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 1,log=True),
        'n_estimators': trial.suggest_int('n_estimators', 750, 1000),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0,log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-1, 1),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1, 2),
    }

    # Determine number of PCA components
    n_components_ratio = trial.suggest_categorical('n_components_ratio', [0.95, 0.99, 1])
    pca = PCA(n_components=n_components_ratio)
    X_pca_train = pca.fit_transform(X_train)
    X_pca_valid = pca.transform(X_valid)
    
    # Train the model
    model = lgb.LGBMClassifier(**param)
    model.fit(X_pca_train, y_train, eval_set=[(X_pca_valid, y_valid)], callbacks=[lgb.early_stopping(stopping_rounds=100)])

    # Predict and calculate AUC
    y_pred_proba = model.predict_proba(X_pca_valid)[:, 1]
    auc = roc_auc_score(y_valid, y_pred_proba)
    
    return auc

# Create the study and optimize
study = optuna.create_study(direction='maximize')
n_trials=100

with tqdm(total=n_trials) as pbar:
    def callback(study, trial):
        pbar.update(1)
    
    study.optimize(objective, n_trials=n_trials, callbacks=[callback])

# Print the best parameters and best score
best_params = study.best_params
best_auc_score = study.best_value
print(f"Best parameters found: {best_params}")
print(f"Best AUC score: {best_auc_score:.2f}")

# Evaluate the best model with metrics
pca = PCA(n_components=best_params['n_components_ratio'])
X_pca_train = pca.fit_transform(X_train)
X_pca_valid = pca.transform(X_valid)
X_pca_test = pca.transform(X_test)

clf = lgb.LGBMClassifier(
    max_depth=best_params['max_depth'],
    num_leaves=best_params['num_leaves'],
    learning_rate=best_params['learning_rate'],
    n_estimators=best_params['n_estimators'],
    reg_alpha=best_params['reg_alpha'],
    reg_lambda=best_params['reg_lambda'],
    scale_pos_weight=best_params['scale_pos_weight']
)
clf.fit(
    X_pca_train,
    y_train,
    eval_set=(X_pca_valid, y_valid),
    callbacks=[lgb.early_stopping(stopping_rounds=100),lgb.log_evaluation(10)]
)

# Predictions and evaluation
y_pred = clf.predict(X_pca_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Contingency table and heatmap
contingency_table = pd.crosstab(pd.Series(y_test, name="Actual"), pd.Series(y_pred, name="Predicted"))
sns.heatmap(contingency_table, annot=True, fmt="d", cmap="Blues")
plt.title("Contingency Table")
plt.show()

# ROC Curve
y_pred_proba = clf.predict_proba(X_pca_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
auc_score = roc_auc_score(y_test, y_pred_proba)

plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {auc_score:.2f}")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.legend(loc="lower right")
plt.show()

print("Best AUC Score:", best_auc_score)
print("Best Parameters:", best_params)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Assuming you already have your dataset as a DataFrame
# Replace this with your actual data loading process
data = {
    "y_true": y_test,
    "y_pred_proba": y_pred_proba
}
df = pd.DataFrame(data)

# Define the number of bins
num_bins = 20
bins = np.linspace(0, 1, num_bins + 1)

# Separate data based on y_true
probas_0 = df[df['y_true'] == 0]['y_pred_proba']
probas_1 = df[df['y_true'] == 1]['y_pred_proba']

# Create histograms for each y_true value
hist_0, _ = np.histogram(probas_0, bins=bins)
hist_1, _ = np.histogram(probas_1, bins=bins)

# Plot the histogram
bar_width = (bins[1] - bins[0]) * 0.4  # Narrower bars for clarity
bin_centers = (bins[:-1] + bins[1:]) / 2

plt.figure(figsize=(12, 6))
plt.bar(bin_centers - bar_width / 2, hist_0, width=bar_width, label='y_true = 0', alpha=0.7, color='blue')
plt.bar(bin_centers + bar_width / 2, hist_1, width=bar_width, label='y_true = 1', alpha=0.7, color='orange')

# Add labels and legend
plt.xlabel('Predicted Probability')
plt.ylabel('Count')
plt.title('Histogram of Predicted Probabilities by True Label')
plt.legend()
plt.xticks(bins)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Show the plot
plt.tight_layout()
plt.show()